# Kaggle shim: one training run

Never develop in this notebook. It is re-run from the top in a fresh container, with no state carried over (plan §12.3).
Change the code in the repository, pin the commit below, and pin the Kaggle environment (torch, CUDA and driver) so that every run of the study shares it.
The cell only clones, installs, starts a raw `nvidia-smi` utilisation log and calls `scripts.train`; the run is idempotent and resumes from its checkpoints with no input.


In [ ]:
# PLACEHOLDERS, set before committing. Set GPU_SHARED and EXPECTED_BACKEND by hand for each job.
REPO_URL, PINNED_COMMIT, DATASET_NAME, RUN_SPEC = "https://github.com/OWNER/attention-benchmark.git", "PINNED_COMMIT_SHA", "DATASET_NAME", "configs/runs/example_mha_s1.yaml"
GPU_SHARED, EXPECTED_BACKEND = "PLACEHOLDER", "PLACEHOLDER"  # yes|no; one of flash, efficient, math, cudnn (plan §2.3, §10.6)
import os, shutil, subprocess as sp, time; REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO): sp.run(["git", "clone", REPO_URL, REPO], check=True)
sp.run(["git", "-C", REPO, "fetch", "origin"], check=True); sp.run(["git", "-C", REPO, "checkout", PINNED_COMMIT], check=True)  # every run, never a stale tree
sp.run(["pip", "install", "-e", REPO, "--no-deps"], check=True)  # --no-deps: keep Kaggle's pinned torch
os.makedirs("/kaggle/working/out/logs", exist_ok=True); log = sp.Popen(["nvidia-smi", "--query-gpu=timestamp,index,utilization.gpu,memory.used", "--format=csv", "-l", "10"], stdout=open(f"/kaggle/working/out/logs/gpu_util_{int(time.time())}.csv", "a")) if shutil.which("nvidia-smi") else None  # raw utilisation log (plan §18, A17); nothing reads it
try: sp.run(["python", "-m", "scripts.train", RUN_SPEC, "--data", f"/kaggle/input/{DATASET_NAME}", "--out", "/kaggle/working/out", "--gpu-shared", GPU_SHARED, "--expected-backend", EXPECTED_BACKEND], cwd=REPO, check=True)
finally: log and log.terminate()